# 02 – Analysis of the FinQA results

**Seminar paper:** Benchmarks for Financial LLM Agents. This notebook covers the empirical part (step 2 of 2).

Reads the raw answers from `results/*.jsonl` (produced by `01_run_experiments.ipynb`) and computes all results. No model calls; runs in seconds.

**Settings**
- **FinBen protocol** (`finben`): FinBen's exact prompt, no system prompt, direct answer. The replication setting.
- **Chain-of-thought** (`cot`) and **Tool use** (`tool`): our own prompts.

**Scoring rules**, from strictest to most lenient:
1. **FinBen EM**: FinBen's own rule from their code (PIXIU, `src/tasks/flare.py`): `acc = 1.0 if results[0].strip() == gold else 0.0`. The *whole* response must equal FinBen's gold answer string character for character. Only meaningful for the FinBen-protocol setting. **These are the numbers that are directly comparable with FinBen's Table 3.**
2. **Strict number match**: only the final answer is used; the number exactly as written must be within 1% of the exact gold result. No rescaling (`14.46%` ≠ `0.14464`), and the rounded human-written answer doesn't count either (`0.14` ≠ `0.14464`).
3. **Tolerant number match**: like strict, but also accepts the rounded human-written answer and percent/decimal forms either way (`14%` = `0.14` = `14`).

Every strict match is also a tolerant match. FinBen EM is usually the strictest, but its gold strings are rounded, so a response matching FinBen's gold can occasionally fail the strict rule.

Applying all three rules to the *same* responses shows how much a benchmark score depends on the scoring rule rather than on the model's reasoning.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from scipy.stats import binomtest

import finqa_eval as fe

MODELS = ["qwen3.5:9b", "gemma4:26b", "qwen3-coder:30b"]
SETTINGS = ["finben", "cot", "tool"]
SETTING_LABEL = {"finben": "FinBen protocol", "cot": "Chain-of-thought", "tool": "Tool use (Python)"}
SETTING_SHORT = {"finben": "FinBen", "cot": "CoT", "tool": "Tool use"}
METRIC_LABEL = {"finben_em": "FinBen EM", "strict": "Strict", "tolerant": "Tolerant"}
RESULTS_DIR = Path("results")
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
TAB_DIR = Path("tables"); TAB_DIR.mkdir(exist_ok=True)

# Colors: one per setting (fixed order), gray for FinBen's published models,
# and one hue light -> dark for the three scoring rules (ordered by strictness)
COLORS = {"finben": "#1baf7a", "cot": "#2a78d6", "tool": "#eb6834", "paper": "#a3a29c"}
METRIC_COLORS = {"finben_em": "#0b5d40", "strict": "#1baf7a", "tolerant": "#9fdcc4"}
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.edgecolor": "#8a8984", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "legend.frameon": False,
})

### LaTeX export

Every table is also saved as a LaTeX file in `tables/` (booktabs style). In the paper:

```latex
\usepackage{booktabs}   % in the preamble
...
\input{tables/table1_accuracy.tex}
```

Figures are saved as vector PDFs in `figures/` and included with `\includegraphics{figures/fig1_accuracy.pdf}`.

In [ ]:
def tex_escape(x):
    s = str(x)
    for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("_", r"\_"), ("#", r"\#"), ("$", r"\$")]:
        s = s.replace(a, b)
    return s

def to_latex(table, path, caption, label, align=None, note=None, raw_cols=()):
    """Write a DataFrame as a booktabs LaTeX table (no extra packages besides booktabs).
    Columns in raw_cols are written as-is (for cells that already contain LaTeX)."""
    cols = list(table.columns)
    align = align or "l" + "r" * (len(cols) - 1)
    lines = [r"\begin{table}[ht]", r"\centering", r"\small", r"\setlength{\tabcolsep}{4pt}",
             rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             rf"\begin{{tabular}}{{{align}}}", r"\toprule",
             " & ".join(tex_escape(c) for c in cols) + r" \\", r"\midrule"]
    for _, row in table.iterrows():
        cells = [str(v) if c in raw_cols else tex_escape(v) for c, v in zip(cols, row)]
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    if note:
        lines.append(rf"\par\smallskip\parbox{{0.95\linewidth}}{{\footnotesize {note}}}")
    lines.append(r"\end{table}")
    Path(path).write_text("\n".join(lines) + "\n")
    print(f"saved {path}")

def tt(model):
    return r"\texttt{" + tex_escape(model) + "}"

def fmt_ci(v, lo, hi, bold=False):
    if pd.isna(v):
        return "--"
    s = f"{v:.2f} [{lo:.2f}, {hi:.2f}]"
    return r"\textbf{" + s + "}" if bold else s

## Load and score

In [ ]:
# Original FinQA (exact gold results) linked to FinBen's version (FinBen's gold strings)
items = {it["id"]: it for it in fe.attach_finben(fe.load_finqa("data"), fe.load_flare_finqa("data"))}

records = []
for path in sorted(RESULTS_DIR.glob("*.jsonl")):
    records += [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
df = pd.DataFrame(records)

# FinBen's rule only applies to the FinBen-protocol setting (our prompts ask for reasoning or code)
df["finben_em"] = [fe.score_finben(r, items[i]) if s == "finben" else np.nan
                   for r, i, s in zip(df["response"], df["id"], df["setting"])]
df["strict"] = [fe.score_strict(f, items[i]) for f, i in zip(df["final"], df["id"])]
df["tolerant"] = [fe.score_tolerant(f, items[i]) for f, i in zip(df["final"], df["id"])]
df["no_answer"] = df["final"].isna()
df["finben_gold"] = df["id"].map(lambda i: items[i]["finben_gold"])
df["gold_answer"] = df["id"].map(lambda i: items[i]["gold_answer"])
df["gold_exe"] = df["id"].map(lambda i: items[i]["gold_exe"])
df["question"] = df["id"].map(lambda i: items[i]["question"])
df["model"] = pd.Categorical(df["model"], MODELS, ordered=True)
df["setting"] = pd.Categorical(df["setting"], SETTINGS, ordered=True)

df.groupby(["model", "setting"], observed=True).size().rename("n questions").to_frame()

## Table 1: Accuracy with 95% bootstrap confidence intervals

In [ ]:
def bootstrap_ci(x, n_boot=5000, seed=0):
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n_boot, len(x))).mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

rows = []
for (model, setting), g in df.groupby(["model", "setting"], observed=True):
    row = {"model": model, "setting": setting, "n": len(g)}
    for metric in ["finben_em", "strict", "tolerant"]:
        x = g[metric].dropna()
        if len(x):
            row[metric] = x.astype(float).mean()
            row[f"{metric}_lo"], row[f"{metric}_hi"] = bootstrap_ci(x)
        else:
            row[metric] = row[f"{metric}_lo"] = row[f"{metric}_hi"] = np.nan
    row["no answer"] = g["no_answer"].mean()
    row["s/question"] = g["seconds"].mean()
    rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(TAB_DIR / "table1_accuracy.csv", index=False)

# LaTeX version: best value per metric in bold, model name only on the first row of each group
t1 = pd.DataFrame({
    "Model": [tt(m) if m != prev else "" for m, prev in zip(summary["model"], summary["model"].shift())],
    "Setting": summary["setting"].map(SETTING_SHORT),
    **{METRIC_LABEL[m]: [fmt_ci(getattr(r, m), getattr(r, f"{m}_lo"), getattr(r, f"{m}_hi"),
                                getattr(r, m) == summary[m].max()) for r in summary.itertuples()]
       for m in ["finben_em", "strict", "tolerant"]},
    "No ans.": summary["no answer"].map("{:.0%}".format),
    "s/q": summary["s/question"].map("{:.1f}".format),
})
to_latex(t1, TAB_DIR / "table1_accuracy.tex",
         caption=f"FinQA accuracy of recent local LLMs ({summary['n'].iloc[0]} test questions) with 95\\% bootstrap confidence intervals.",
         label="tab:finqa_accuracy", align="llrrrrr", raw_cols={"Model", "FinBen EM", "Strict", "Tolerant"},
         note="Settings: FinBen: FinBen's prompt and direct answer; CoT: chain-of-thought; Tool use: the model writes Python that is executed. "
              "FinBen EM: FinBen's rule, the whole response must equal the gold string (only defined for the FinBen protocol). "
              "Strict: the final number as written is within 1\\% of the exact result. "
              "Tolerant: additionally accepts the rounded human-written answer and percentage/decimal forms either way. "
              "No ans.: no parseable final answer. s/q: mean seconds per question on a 24\\,GB laptop.")

show = summary.copy()
for m in ["finben_em", "strict", "tolerant"]:
    show[m] = [fmt_ci(v, lo, hi) for v, lo, hi in zip(show[m], show[f"{m}_lo"], show[f"{m}_hi"])]
show[["model", "setting", "n", "finben_em", "strict", "tolerant", "no answer", "s/question"]].round(2)

## Figure 1: Comparison with FinBen (Table 3 of Xie et al., 2024)

**Directly comparable:** our models under the FinBen protocol and FinBen's scoring rule, next to FinBen's published FinQA results (zero-shot EmAcc). Same test set, same prompt, same scoring rule.

Remaining differences, to state in the paper: 200-question random sample instead of all 1,147 (hence the confidence intervals); local quantized models via Ollama with temperature 0; values marked * were taken by FinBen from earlier papers rather than run with their code.

In [ ]:
finben_paper = pd.Series({
    "GPT-4*": 0.63, "ChatGPT*": 0.58, "LLaMA2-70B": 0.06, "FinMA-7B": 0.04, "Gemini": 0.00,
    "LLaMA2-7B-chat": 0.00, "LLaMA3-8B": 0.00, "FinGPT-7B": 0.00, "InternLM-7B": 0.00,
    "Falcon-7B": 0.00, "Mixtral-7B": 0.00, "CFGPT-7B": 0.00,
})  # FinBen, Table 3, FinQA EmAcc

ours = summary[summary["setting"] == "finben"].set_index("model")
comp = pd.concat([
    pd.DataFrame({"accuracy": finben_paper, "lo": np.nan, "hi": np.nan, "source": "FinBen (2024)"}),
    pd.DataFrame({"accuracy": ours["finben_em"].values, "lo": ours["finben_em_lo"].values,
                  "hi": ours["finben_em_hi"].values, "source": "This study"}, index=ours.index.astype(str)),
]).sort_values("accuracy")
comp.to_csv(TAB_DIR / "table2_finben_comparison.csv")

fig, ax = plt.subplots(figsize=(7, 5))
is_ours = comp["source"] == "This study"
ax.barh(comp.index, comp["accuracy"], height=0.7, color=[COLORS["finben"] if o else COLORS["paper"] for o in is_ours])
y_ours = np.where(is_ours)[0]
ax.errorbar(comp["accuracy"][is_ours], y_ours, xerr=[comp["accuracy"][is_ours] - comp["lo"][is_ours],
            comp["hi"][is_ours] - comp["accuracy"][is_ours]], fmt="none", ecolor="#52514e", elinewidth=1, capsize=3)
for i, (v, hi) in enumerate(zip(comp["accuracy"], comp["hi"])):
    ax.text((hi if pd.notna(hi) else v) + 0.01, i, f"{v:.2f}", va="center", fontsize=8, color="#52514e")
ax.set_xlim(0, 1); ax.set_xlabel("FinQA accuracy, FinBen exact match (whole response = gold answer)")
ax.grid(axis="x", color="#e6e5e0"); ax.grid(axis="y", visible=False)
for lbl in ax.get_yticklabels():
    if lbl.get_text() in MODELS:
        lbl.set_fontfamily("monospace")
ax.legend(handles=[Patch(color=COLORS["paper"], label="FinBen (2024), earlier LLMs"),
                   Patch(color=COLORS["finben"], label="This study, same protocol (95% CI)")], loc="lower right")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig1_finben_comparison.pdf"); plt.show()

t2 = comp.sort_values("accuracy", ascending=False)
t2 = pd.DataFrame({
    "Model": [tt(m) if src == "This study" else tex_escape(m) for m, src in zip(t2.index, t2["source"])],
    "Source": t2["source"],
    "FinQA EM": [f"{v:.2f}" if pd.isna(lo) else f"{v:.2f} [{lo:.2f}, {hi:.2f}]"
                 for v, lo, hi in zip(t2["accuracy"], t2["lo"], t2["hi"])],
})
to_latex(t2, TAB_DIR / "table2_finben_comparison.tex",
         caption="FinQA accuracy under FinBen's protocol and exact-match rule: models in this study compared with FinBen.",
         label="tab:finben_comparison", align="llr", raw_cols={"Model"},
         note="FinBen values from Xie et al.\\ (2024), Table 3 (zero-shot, all 1{,}147 test questions); * taken by FinBen from earlier papers. "
              "This study: same prompt and scoring rule, 200-question random sample, 95\\% bootstrap confidence intervals.")

## Figure 2: How much does the scoring rule matter?

The same FinBen-protocol responses, scored with all three rules. The gap between FinBen EM and the tolerant rule is the part of the score that depends on answer formatting rather than on getting the number right.

In [ ]:
fb = summary[summary["setting"] == "finben"].set_index("model").reindex(MODELS)
metrics = ["finben_em", "strict", "tolerant"]
fig, ax = plt.subplots(figsize=(7.5, 3.8))
x = np.arange(len(MODELS)); w = 0.26
for k, m in enumerate(metrics):
    pos = x + (k - 1) * (w + 0.02)
    ax.bar(pos, fb[m], w, color=METRIC_COLORS[m], label=METRIC_LABEL[m] + {"finben_em": " (FinBen's rule)", "strict": " number match", "tolerant": " number match"}[m])
    ax.errorbar(pos, fb[m], yerr=[fb[m] - fb[f"{m}_lo"], fb[f"{m}_hi"] - fb[m]], fmt="none", ecolor="#52514e", elinewidth=1, capsize=3)
    for p, v, hi in zip(pos, fb[m], fb[f"{m}_hi"]):
        ax.text(p, hi + 0.02, f"{v:.2f}", ha="center", fontsize=8, color="#52514e")
ax.set_xticks(x, MODELS); ax.set_ylim(0, 1); ax.set_ylabel("Accuracy (FinBen-protocol responses)")
ax.legend(loc="upper left", ncols=3)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig2_scoring_rules.pdf"); plt.show()

fb[metrics].round(2)

## Figure 3: Does reasoning or tool use help?

All three settings, scored with the strict and the tolerant number rule (FinBen's whole-response rule cannot be applied to responses that contain reasoning or code).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
x = np.arange(len(MODELS)); w = 0.26
for ax, metric, title in zip(axes, ["strict", "tolerant"], ["Strict number match", "Tolerant number match"]):
    for k, setting in enumerate(SETTINGS):
        s = summary[summary["setting"] == setting].set_index("model").reindex(MODELS)
        pos = x + (k - 1) * (w + 0.02)
        ax.bar(pos, s[metric], w, color=COLORS[setting], label=SETTING_LABEL[setting])
        ax.errorbar(pos, s[metric], yerr=[s[metric] - s[f"{metric}_lo"], s[f"{metric}_hi"] - s[metric]],
                    fmt="none", ecolor="#52514e", elinewidth=1, capsize=3)
        for p, v, hi in zip(pos, s[metric], s[f"{metric}_hi"]):
            ax.text(p, hi + 0.02, f"{v:.2f}", ha="center", fontsize=7.5, color="#52514e")
    ax.set_xticks(x, MODELS); ax.set_title(title, fontsize=10, loc="left"); ax.set_ylim(0, 1)
axes[0].set_ylabel("Accuracy (200 FinQA questions)")
axes[0].legend(loc="upper left")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig3_settings.pdf"); plt.show()

## Table 3: Paired tests (McNemar, exact)

Every model and setting answered the *same* 200 questions, so comparisons are paired: only the questions where exactly one of the two is correct matter. Tolerant scoring, so all settings are compared on the same rule.

In [ ]:
def mcnemar(a, b):
    # a, b: boolean Series indexed by question id (same questions)
    a_only, b_only = int((a & ~b).sum()), int((~a & b).sum())
    p = binomtest(a_only, a_only + b_only, 0.5).pvalue if a_only + b_only else 1.0
    return a_only, b_only, p

wide = df.pivot_table(index="id", columns=["model", "setting"], values="tolerant", observed=True).astype(bool)
tests = []
for m in MODELS:                       # settings against each other, per model
    for s1, s2 in [("cot", "finben"), ("tool", "finben"), ("tool", "cot")]:
        if (m, s1) in wide and (m, s2) in wide:
            a, b, p = mcnemar(wide[(m, s1)], wide[(m, s2)])
            tests.append({"comparison": f"{m}: {SETTING_SHORT[s1]} vs. {SETTING_SHORT[s2]}",
                          "only first correct": a, "only second correct": b, "p-value": p})
for s in SETTINGS:                     # models against each other, per setting
    for i, m1 in enumerate(MODELS):
        for m2 in MODELS[i + 1:]:
            if (m1, s) in wide and (m2, s) in wide:
                a, b, p = mcnemar(wide[(m1, s)], wide[(m2, s)])
                tests.append({"comparison": f"{SETTING_SHORT[s]}: {m1} vs. {m2}",
                              "only first correct": a, "only second correct": b, "p-value": p})
tests = pd.DataFrame(tests)
tests.to_csv(TAB_DIR / "table3_mcnemar.csv", index=False)

if len(tests):
    t3 = pd.DataFrame({
        "Comparison (A vs. B)": tests["comparison"],
        "Only A correct": tests["only first correct"],
        "Only B correct": tests["only second correct"],
        "p-value": tests["p-value"].map(lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"),
    })
    to_latex(t3, TAB_DIR / "table3_mcnemar.tex",
             caption="Paired comparisons on the same FinQA questions (exact McNemar test, tolerant scoring).",
             label="tab:mcnemar", align="lrrr")
tests.round(4)

## Tool setting: how often did the code run, and did the repair round help?

In [ ]:
tool = df[df["setting"] == "tool"]
tool_stats = tool.groupby("model", observed=True).agg(
    n=("id", "size"),
    first_try_ok=("n_repairs", lambda r: (r == 0).mean()),
    needed_repair=("n_repairs", "mean"),
    failed_after_repair=("exec_error", lambda e: e.notna().mean()),
    accuracy_tolerant=("tolerant", "mean"),
)
rep = tool[tool["n_repairs"] == 1]
tool_stats["repair_success_rate"] = rep.groupby("model", observed=True)["exec_error"].apply(lambda e: e.isna().mean())

t4 = pd.DataFrame({
    "Model": [tt(m) for m in tool_stats.index],
    "Ran first try": tool_stats["first_try_ok"].map("{:.0%}".format),
    "Repair needed": tool_stats["needed_repair"].map("{:.0%}".format),
    "Repair fixed it": tool_stats["repair_success_rate"].map(lambda v: "--" if pd.isna(v) else f"{v:.0%}"),
    "Still failing": tool_stats["failed_after_repair"].map("{:.0%}".format),
    "Acc. (tol.)": tool_stats["accuracy_tolerant"].map("{:.2f}".format),
})
to_latex(t4, TAB_DIR / "table4_tool_use.tex",
         caption="Tool-use setting: execution of model-written Python code and the effect of one repair round.",
         label="tab:tool_use", align="lrrrrr", raw_cols={"Model"})
tool_stats.round(2)

## Speed

In [ ]:
speed = df.groupby(["model", "setting"], observed=True).agg(
    seconds_mean=("seconds", "mean"), seconds_median=("seconds", "median"),
    output_tokens_mean=("output_tokens", "mean"), prompt_tokens_mean=("prompt_tokens", "mean"),
    truncated_prompts=("truncated_prompt", "sum"),
)
speed.round(1)

## FinBen protocol: what do the responses look like?

Typical reasons why a response fails FinBen's rule although the number is right (e.g. an added sentence, a unit, a different rounding). Useful examples for the limitations section.

In [ ]:
fb_resp = df[df["setting"] == "finben"]
near_miss = fb_resp[(fb_resp["finben_em"] == 0) & fb_resp["tolerant"]]
print(f"Right number but failed FinBen's rule: {len(near_miss)} of {len(fb_resp)} FinBen-protocol responses\n")
for _, r in near_miss.groupby("model", observed=True).head(2).iterrows():
    print(f"[{r['model']}] gold: {r['finben_gold']!r}\n  response: {r['response'].strip()[:200]!r}\n")

## Error analysis

1. The cell below exports 30 random wrong answers (tolerant scoring) for each of the chain-of-thought and tool settings to `tables/error_sample.csv`.
2. Open the file and fill in the `category` column by hand, with one of: `extraction` (wrong number taken from table or text), `reasoning` (wrong formula or steps), `arithmetic` (right formula, wrong calculation), `scale/sign` (unit, %, sign or magnitude), `format` (no or unparseable answer), `gold` (the gold answer itself looks wrong).
3. Save it as `tables/error_sample_labeled.csv` and run the following cell to get the figure.

The export only runs if the labeled file doesn't exist yet, so your labels are never overwritten.

In [ ]:
ERROR_SETTINGS = ["cot", "tool"]
labeled = TAB_DIR / "error_sample_labeled.csv"
if not labeled.exists():
    wrong = df[~df["tolerant"]]
    err = pd.concat([wrong[wrong["setting"] == s].pipe(lambda g: g.sample(min(30, len(g)), random_state=0))
                     for s in ERROR_SETTINGS])
    err = err[["id", "model", "setting", "question", "gold_answer", "gold_exe", "final", "response", "code"]].assign(category="")
    err.to_csv(TAB_DIR / "error_sample.csv", index=False)
    print(f"Exported {len(err)} errors to tables/error_sample.csv – label them, save as error_sample_labeled.csv")

In [ ]:
if labeled.exists():
    lab = pd.read_csv(labeled)
    counts = pd.crosstab(lab["category"], lab["setting"]).reindex(columns=ERROR_SETTINGS, fill_value=0)
    counts = counts.loc[counts.sum(axis=1).sort_values().index]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    y = np.arange(len(counts)); h = 0.38
    for k, s in enumerate(ERROR_SETTINGS):
        ax.barh(y + (k - 0.5) * (h + 0.02), counts[s], h, color=COLORS[s], label=SETTING_LABEL[s])
    ax.set_yticks(y, counts.index); ax.set_xlabel("Number of errors (hand-labeled sample)")
    ax.grid(axis="x", color="#e6e5e0"); ax.grid(axis="y", visible=False); ax.legend()
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig4_error_categories.pdf"); plt.show()
    t5 = counts.iloc[::-1].reset_index().rename(columns={"category": "Error category", **SETTING_LABEL})
    to_latex(t5, TAB_DIR / "table5_error_categories.tex",
             caption="Error categories in a hand-labeled sample of wrong answers.",
             label="tab:errors", align="l" + "r" * len(ERROR_SETTINGS))
    display(counts)
else:
    print("No labeled file yet.")

## Examples for the paper

One question that tool use solved but chain-of-thought didn't (and the other way round): useful as illustrations.

In [ ]:
m = MODELS[-1]
if (m, "tool") in wide and (m, "cot") in wide:
    for label, mask in [("tool right, cot wrong", wide[(m, "tool")] & ~wide[(m, "cot")]),
                        ("cot right, tool wrong", ~wide[(m, "tool")] & wide[(m, "cot")])]:
        ids = mask[mask].index
        if len(ids):
            qid = ids[0]
            print(f"=== {label} ({m}) — {qid}\nQ: {items[qid]['question']}\nGold: {items[qid]['gold_answer']} ({items[qid]['gold_exe']})")
            for s in ["cot", "tool"]:
                r = df[(df["id"] == qid) & (df["model"] == m) & (df["setting"] == s)].iloc[0]
                print(f"--- {s}: final = {r['final']}\n{(r['code'] if s == 'tool' else r['response'])[-700:]}")
            print()